## Save Yearly Averages
- Creates a new DataSet for Yearly Averages Separately (with all load types)

In [7]:
import pandas as pd
from pathlib import Path
import duckdb   
                                                                                                                                                                                                                  
DATA_DIR = next(
    p / "data" for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data").exists()
)

usa_dir = DATA_DIR / "processed" / "usa_data"
db_path = DATA_DIR / "processed" / "usa_data" / "pjm_lmp_historical.duckdb"                                                                                                                
db_path_2yr = DATA_DIR / "processed" / "usa_data" / "pjm_lmp_va_2yr.duckdb" 


In [2]:
# Load all four CSVs
hist_load = pd.read_csv(usa_dir / "va_lmp_historical_monthly_avg_load.csv")
hist_all  = pd.read_csv(usa_dir / "va_lmp_historical_monthly_avg_all.csv")
std_load  = pd.read_csv(usa_dir / "va_lmp_monthly_avg_load.csv")
std_all   = pd.read_csv(usa_dir / "va_lmp_monthly_avg_all.csv")

print("Historical LOAD:", hist_load.shape)
print("Historical ALL: ", hist_all.shape)
print("Standard LOAD:  ", std_load.shape)
print("Standard ALL:   ", std_all.shape)

Historical LOAD: (61562, 9)
Historical ALL:  (77995, 9)
Standard LOAD:   (19894, 9)
Standard ALL:    (25305, 9)


In [13]:
# Combine LOAD only
combined_load = pd.concat([hist_load, std_load], ignore_index=True)
combined_load = combined_load.sort_values(["year", "month", "pnode_id"]).reset_index(drop=True)

print(f"Combined LOAD: {len(combined_load):,} rows")
print(f"Year range: {combined_load['year'].min()} – {combined_load['year'].max()}")
print(f"Unique pnode_ids: {combined_load['pnode_id'].nunique()}")

combined_load.to_csv(usa_dir / "va_lmp_combined_load_monthly.csv", index=False)
print("Saved va_lmp_combined_load_monthly.csv")

Combined LOAD: 81,456 rows
Year range: 2018 – 2026
Unique pnode_ids: 899
Saved va_lmp_combined_load_monthly.csv


In [14]:
# Combine all types - for monthly data
combined_all = pd.concat([hist_all, std_all], ignore_index=True)
combined_all = combined_all.sort_values(["year", "month", "pnode_id", "type"]).reset_index(drop=True)

print(f"Combined ALL: {len(combined_all):,} rows")
print(f"Year range: {combined_all['year'].min()} – {combined_all['year'].max()}")
print(f"Unique pnode_ids: {combined_all['pnode_id'].nunique()}")
print(f"\nType breakdown:")
print(combined_all["type"].value_counts())

combined_all.to_csv(usa_dir / "va_lmp_combined_all_monthly.csv", index=False)
print("\nSaved va_lmp_combined_all_monthly.csv")

Combined ALL: 103,300 rows
Year range: 2018 – 2026
Unique pnode_ids: 1139

Type breakdown:
type
LOAD    81456
GEN     18280
EHV      3564
Name: count, dtype: int64

Saved va_lmp_combined_all_monthly.csv


In [5]:
# Verify no overlap between historical and standard date ranges
hist_months = set(zip(hist_load["year"], hist_load["month"]))
std_months  = set(zip(std_load["year"], std_load["month"]))
overlap = hist_months & std_months

if overlap:
    print(f"WARNING: {len(overlap)} overlapping months found: {sorted(overlap)}")
else:
    print("No overlapping months — clean merge")

No overlapping months — clean merge


In [11]:
#Export yearly averages to CSV, combining historical and standard DBs for 2018-2026
#2024 has a slightly different treament - as it is split between the two DBs, we pull it separately and combine before averaging
con_hist = duckdb.connect(str(db_path))                                                                                                                                                    
con_std = duckdb.connect(str(db_path_2yr))                                                                                                                                                 
                                                                                                                                                                                            
# 2024: pull from both DBs and combine                                                                                                                                                     
hist_2024 = con_hist.execute("""                                                                                                                                                           
    SELECT datetime_beginning_ept, pnode_id, pnode_name, type,                                                                                                                             
            total_lmp_rt, system_energy_price_rt, congestion_price_rt, marginal_loss_price_rt                                                                                               
    FROM rt_hrl_lmps                        
    WHERE substr(datetime_beginning_ept, 1, 4) = '2024'                                                                                                                                    
""").fetchdf()                              
                                                                                                                                                                                            
std_2024 = con_std.execute("""
    SELECT datetime_beginning_ept, pnode_id, pnode_name, type,                                                                                                                             
            total_lmp_rt, system_energy_price_rt, congestion_price_rt, marginal_loss_price_rt
    FROM rt_hrl_lmps                                                                                                                                                                       
    WHERE substr(datetime_beginning_ept, 1, 4) = '2024'
""").fetchdf()                              
                                                                                                                                                                                            
raw_2024 = pd.concat([hist_2024, std_2024], ignore_index=True)
                                                                                                                                                                                            
avg_2024 = raw_2024.groupby(["pnode_id", "pnode_name", "type"]).agg(
    avg_total_lmp=("total_lmp_rt", "mean"),                                                                                                                                                
    avg_energy=("system_energy_price_rt", "mean"),
    avg_congestion=("congestion_price_rt", "mean"),                                                                                                                                        
    avg_loss=("marginal_loss_price_rt", "mean"),
    hours=("total_lmp_rt", "count")                                                                                                                                                        
).reset_index()                                                                                                                                                                            
avg_2024["year"] = 2024                 
                                                                                                                                                                                            
# All other years: query each DB directly                                                                                                                                                  
yearly_hist = con_hist.execute("""      
    SELECT substr(datetime_beginning_ept, 1, 4) AS year,                                                                                                                                   
            pnode_id, pnode_name, type,                                                                                                                                                     
            AVG(total_lmp_rt) AS avg_total_lmp,
            AVG(system_energy_price_rt) AS avg_energy,                                                                                                                                      
            AVG(congestion_price_rt) AS avg_congestion,
            AVG(marginal_loss_price_rt) AS avg_loss,                                                                                                                                        
            COUNT(*) AS hours            
    FROM rt_hrl_lmps                                                                                                                                                                       
    WHERE substr(datetime_beginning_ept, 1, 4) BETWEEN '2018' AND '2023'                                                                                                                   
    GROUP BY year, pnode_id, pnode_name, type
""").fetchdf()                                                                                                                                                                             
                                            
yearly_std = con_std.execute("""                                                                                                                                                           
    SELECT substr(datetime_beginning_ept, 1, 4) AS year,
            pnode_id, pnode_name, type,                                                                                                                                                     
            AVG(total_lmp_rt) AS avg_total_lmp,
            AVG(system_energy_price_rt) AS avg_energy,                                                                                                                                      
            AVG(congestion_price_rt) AS avg_congestion,
            AVG(marginal_loss_price_rt) AS avg_loss,
            COUNT(*) AS hours                                                                                                                                                               
    FROM rt_hrl_lmps                    
    WHERE substr(datetime_beginning_ept, 1, 4) = '2025'                                                                                                                         
    GROUP BY year, pnode_id, pnode_name, type                                                                                                                                              
""").fetchdf()                          
                                                                                                                                                                                            
con_hist.close()                                                                                                                                                                           
con_std.close()

# Combine everything
yearly_hist["year"] = yearly_hist["year"].astype(int)
yearly_std["year"] = yearly_std["year"].astype(int)
avg_2024["year"] = 2024

yearly_true = pd.concat([yearly_hist, avg_2024, yearly_std], ignore_index=True)
yearly_true["year"] = yearly_true["year"].astype(int)
yearly_true = yearly_true.sort_values(["year", "pnode_id", "type"]).reset_index(drop=True)

print(f"Rows: {len(yearly_true):,}")
print(f"Year range: {yearly_true['year'].min()} – {yearly_true['year'].max()}")

yearly_true.to_csv(usa_dir / "va_lmp_yearly_avg_all.csv", index=False)
print("Saved va_lmp_yearly_avg_all.csv")
                                                                                                                                                                                            

Rows: 8,413
Year range: 2018 – 2025
Saved va_lmp_yearly_avg_all.csv
